# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_fe_layer_one_model_30_trials.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_fe_layer_one_model_30_trials.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.005332,0.000921,0.993747,0.001583,0.000348,0.998069,0.007467,0.000788,0.991745
1,0.995298,0.000565,0.004136,0.996461,0.000307,0.003233,0.992663,0.000880,0.006457
2,0.003239,0.001006,0.995755,0.004254,0.000524,0.995222,0.004121,0.001199,0.994680
3,0.004635,0.001771,0.993594,0.003324,0.001662,0.995014,0.007385,0.002109,0.990506
4,0.994943,0.000108,0.004949,0.992094,0.000021,0.007886,0.994977,0.000148,0.004875


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2,cat_0,cat_1,cat_2
0,0.009638,0.004344,0.986018,0.010005,0.004287,0.985708,0.007769,0.003721,0.988511
1,0.522299,0.003439,0.474262,0.480004,0.003110,0.516887,0.509202,0.001904,0.488894
2,0.997536,0.001794,0.000670,0.999363,0.000490,0.000148,0.994896,0.004031,0.001073
3,0.989331,0.000063,0.010606,0.987457,0.000013,0.012530,0.994476,0.000479,0.005046
4,0.008243,0.002363,0.989394,0.011684,0.003013,0.985303,0.007928,0.002671,0.989401


# Machine Learning

In [6]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [7]:
cols_use = ['lgbm_0', 'lgbm_1', 'lgbm_2']

In [8]:
def objective(trial, X, y):

    w0 = trial.suggest_float('weight_class_0', 0.0, 10.0, log=False)
    w1 = trial.suggest_float('weight_class_1', 0.0, 10.0, log=False)
    w2 = trial.suggest_float('weight_class_2', 0.0, 10.0, log=False)
    weights = np.array([w0, w1, w2])

    proba = X.loc[:, cols_use].to_numpy()
    
    weighted_probas = proba * weights
    pred = np.argmax(weighted_probas, axis=1)
    
    score = balanced_accuracy_score(y, pred)

    return score


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=3000, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-10 16:12:06,763] A new study created in memory with name: no-name-b0cf5a24-946d-4aee-947f-82037a085b5e
                                                                                                                                                                                                                   

[I 2026-07-10 16:12:06,933] Trial 5 finished with value: 0.8687846647110424 and parameters: {'weight_class_0': 2.2605978459710165, 'weight_class_1': 0.20858621258991628, 'weight_class_2': 5.788328507412893}. Best is trial 5 with value: 0.8687846647110424.
[I 2026-07-10 16:12:06,940] Trial 3 finished with value: 0.9084220434519971 and parameters: {'weight_class_0': 3.5741421468466816, 'weight_class_1': 7.8698227110758046, 'weight_class_2': 4.551571179132798}. Best is trial 3 with value: 0.9084220434519971.
[I 2026-07-10 16:12:06,947] Trial 0 finished with value: 0.8265867697917612 and parameters: {'weight_class_0': 8.37742777532354, 'weight_class_1': 0.6455653071284184, 'weight_class_2': 5.499711537559147}. Best is trial 3 with value: 0.9084220434519971.
[I 2026-07-10 16:12:06,948] Trial 10 finished with value: 0.8469858221956109 and parameters: {'weight_class_0': 6.626153401003126, 'weight_class_1': 2.730356241245999, 'weight_class_2': 3.8468499962747282}. Best is trial 3 with value: 0

[I 2026-07-10 16:12:07,083] Trial 13 finished with value: 0.9161250322862021 and parameters: {'weight_class_0': 3.6340560672697517, 'weight_class_1': 6.514001713560442, 'weight_class_2': 6.627998551828179}. Best is trial 7 with value: 0.9483947788068958.
[I 2026-07-10 16:12:07,103] Trial 14 finished with value: 0.9293253549481371 and parameters: {'weight_class_0': 0.0634752805038643, 'weight_class_1': 4.761119355278575, 'weight_class_2': 9.45941446037954}. Best is trial 7 with value: 0.9483947788068958.
[I 2026-07-10 16:12:07,142] Trial 16 finished with value: 0.9486569845543666 and parameters: {'weight_class_0': 0.3858344094271557, 'weight_class_1': 5.493736673484943, 'weight_class_2': 8.462015703635382}. Best is trial 16 with value: 0.9486569845543666.
[I 2026-07-10 16:12:07,157] Trial 15 finished with value: 0.9022691068563607 and parameters: {'weight_class_0': 0.04368780665012406, 'weight_class_1': 5.462076053625172, 'weight_class_2': 8.532665274886348}. Best is trial 16 with value

Best trial: 25. Best value: 0.948812:   1%|█▌                                                                                                                                    | 36/3000 [00:00<00:56, 52.65it/s]

[I 2026-07-10 16:12:07,272] Trial 24 finished with value: 0.9452222496532752 and parameters: {'weight_class_0': 0.17383542029910348, 'weight_class_1': 9.853384384423055, 'weight_class_2': 8.221301861613984}. Best is trial 16 with value: 0.9486569845543666.
[I 2026-07-10 16:12:07,296] Trial 25 finished with value: 0.9488119903437447 and parameters: {'weight_class_0': 1.1271046771091644, 'weight_class_1': 9.75591048739037, 'weight_class_2': 8.039971225180476}. Best is trial 25 with value: 0.9488119903437447.
[I 2026-07-10 16:12:07,327] Trial 26 finished with value: 0.9360593128904883 and parameters: {'weight_class_0': 1.7478283841818345, 'weight_class_1': 3.8786327228960378, 'weight_class_2': 7.20217112400275}. Best is trial 25 with value: 0.9488119903437447.
[I 2026-07-10 16:12:07,347] Trial 27 finished with value: 0.9412275305426294 and parameters: {'weight_class_0': 1.3439973192421268, 'weight_class_1': 3.738975489489343, 'weight_class_2': 7.4463919890373536}. Best is trial 25 with va

...

[I 2026-07-10 16:13:27,247] Trial 2977 finished with value: 0.9495565495296746 and parameters: {'weight_class_0': 0.5614606258703947, 'weight_class_1': 7.790345083841193, 'weight_class_2': 7.65402645102576}. Best is trial 2787 with value: 0.9497743750090324.
[I 2026-07-10 16:13:27,269] Trial 2979 finished with value: 0.9496111829338645 and parameters: {'weight_class_0': 0.5850772387298694, 'weight_class_1': 7.794958139471504, 'weight_class_2': 7.522230257987794}. Best is trial 2787 with value: 0.9497743750090324.
[I 2026-07-10 16:13:27,326] Trial 2980 finished with value: 0.9496227832834422 and parameters: {'weight_class_0': 0.5781294267688949, 'weight_class_1': 7.824824635607526, 'weight_class_2': 7.542015055280807}. Best is trial 2787 with value: 0.9497743750090324.
[I 2026-07-10 16:13:27,355] Trial 2981 finished with value: 0.9493056485376664 and parameters: {'weight_class_0': 0.8671933554880158, 'weight_class_1': 8.417609305914013, 'weight_class_2': 7.519867647929745}. Best is tria

Best trial: 2787. Best value: 0.949774: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▌| 2990/3000 [01:20<00:00, 27.70it/s]

[I 2026-07-10 16:13:27,503] Trial 2984 finished with value: 0.9491970237759141 and parameters: {'weight_class_0': 0.8641247779795299, 'weight_class_1': 7.75479331650493, 'weight_class_2': 7.523348725792318}. Best is trial 2787 with value: 0.9497743750090324.
[I 2026-07-10 16:13:27,509] Trial 2985 finished with value: 0.9491657905357366 and parameters: {'weight_class_0': 0.8837413543448804, 'weight_class_1': 7.812228595215487, 'weight_class_2': 7.510248662472582}. Best is trial 2787 with value: 0.9497743750090324.
[I 2026-07-10 16:13:27,575] Trial 2987 finished with value: 0.9488888572485962 and parameters: {'weight_class_0': 0.9472061929879934, 'weight_class_1': 7.7858652635010035, 'weight_class_2': 7.488315591699197}. Best is trial 2787 with value: 0.9497743750090324.
[I 2026-07-10 16:13:27,589] Trial 2986 finished with value: 0.9489918589850079 and parameters: {'weight_class_0': 0.9230727638332521, 'weight_class_1': 7.685425704303277, 'weight_class_2': 7.479948295080044}. Best is tri

Best trial: 2787. Best value: 0.949774: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3000/3000 [01:21<00:00, 36.99it/s]

[I 2026-07-10 16:13:27,726] Trial 2992 finished with value: 0.9488574185707193 and parameters: {'weight_class_0': 1.0047401052167664, 'weight_class_1': 8.287895722793179, 'weight_class_2': 7.493497356744003}. Best is trial 2787 with value: 0.9497743750090324.
[I 2026-07-10 16:13:27,734] Trial 2991 finished with value: 0.948860908260765 and parameters: {'weight_class_0': 0.9858064453007207, 'weight_class_1': 8.287015037234713, 'weight_class_2': 7.451002957376147}. Best is trial 2787 with value: 0.9497743750090324.
[I 2026-07-10 16:13:27,769] Trial 2993 finished with value: 0.949207097424455 and parameters: {'weight_class_0': 0.9886949595153631, 'weight_class_1': 8.319839978569364, 'weight_class_2': 9.35219841892416}. Best is trial 2787 with value: 0.9497743750090324.
[I 2026-07-10 16:13:27,793] Trial 2994 finished with value: 0.9488336596566381 and parameters: {'weight_class_0': 1.0193689007547628, 'weight_class_1': 8.314459451398513, 'weight_class_2': 7.424040791576996}. Best is trial 

In [9]:
weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = X_test.loc[:, cols_use].to_numpy() * weights

pred = np.argmax(weighted_probas, axis=1)

In [10]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [11]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.1.5_lightgbm_submission.csv', index=False)

In [12]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
